# Advanced Python Descriptor Problems — A Step-by-Step Tutorial
## Property lookup resolution · a NEW collection of worked problems

This notebook is **not** a reformat of the earlier exercise set. It uses the teaching rhythm of the supplied *Property Lookup Resolution* lesson: first an observable surprise, then a tiny controlled experiment, a careful explanation, one more experiment, and finally an implementable solution with tests. Every problem is split into short markdown and code cells. No external packages or internet access are needed.

**Core source concepts:** data descriptors versus non-data descriptors; same-name keys in an instance `__dict__`; `__set_name__`; correct direct dictionary storage; the danger of recursive `setattr`; and the `__slots__` qualification. **Advanced extensions** are marked as such and go beyond the original short lesson.

**How to use:** run the notebook from the first cell through the last. Each exercise uses unique class names (suffixes such as `P01`), so experiments will not overwrite definitions from other exercises. Try to predict the result at each step before executing it.

## Our working model (read this before Problem 01)

For *ordinary instance reads*, under `object.__getattribute__`, first find the **first matching name in the class MRO**. If that class attribute is a data descriptor with `__get__`, call it before consulting the instance dictionary. Otherwise the instance dictionary may win; if it does not, a found non-data descriptor gets its `__get__` call, and an ordinary class value is returned as-is. Missing names raise `AttributeError`, after which normal dot-lookup machinery may invoke `__getattr__`.

This is a useful model, **not** a universal implementation: custom `__getattribute__`, implicit special-method dispatch, `super()`, metaclass-based class access, and write/delete operations have additional rules. We will investigate several of those boundaries instead of assuming the model applies everywhere.

For clarity, a *data descriptor* supplies `__set__` and/or `__delete__`, normally alongside `__get__`; a *non-data descriptor* supplies `__get__` but neither mutation hook. A descriptor object can be shared on the class; the **values must belong to individual instances**.

In [1]:
from contextlib import contextmanager
from functools import cached_property
import inspect

@contextmanager
def expect_raises(error_type):
    """Assert an expected exception without adding third-party dependencies."""
    try:
        yield
    except error_type:
        pass
    else:
        raise AssertionError(f"Expected {error_type.__name__}")

print("Tutorial helpers loaded.")

Tutorial helpers loaded.


### How the checks are written

We prefer `assert` for invariants, small event logs for dispatch order, and `expect_raises` for negative paths. We avoid timestamps, network requests, and unconstrained recursion, so the full notebook is reproducible. Most examples deliberately mutate `__dict__` to **demonstrate a bypass**, not to recommend doing so in normal application code.

---
# Problem 01 — An apparent contradiction: the same name returns different values

**The challenge.** Construct a data descriptor and a non-data descriptor with matching instance-dictionary entries. Observe which implementation Python actually calls, rather than guessing from `vars(obj)`.

**What you will prove.** Read precedence, class-level inspection, and the difference between finding a descriptor and calling it.

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Write descriptors that leave an audit trail

In [2]:
log01 = []

class Data01:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        log01.append('data.get')
        return 'validated-value'

    def __set__(self, instance, value):
        log01.append(('data.set', value))
        instance.__dict__['locked'] = value

class NonData01:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        log01.append('nondata.get')
        return 'computed-value'

**What just happened?** Both classes have `__get__`. Only `Data01` has `__set__`; that extra method determines read precedence even when its implementation simply writes to the dictionary.

### Step 2: Install both objects on a host class, then bypass assignments

In [3]:
class Host01:
    locked = Data01()
    optional = NonData01()

h01 = Host01()
h01.__dict__.update(locked='raw-locked', optional='raw-optional')
print(vars(h01))
print('class objects:', Host01.locked is vars(Host01)['locked'],
      Host01.optional is vars(Host01)['optional'])

{'locked': 'raw-locked', 'optional': 'raw-optional'}
class objects: True True


**What just happened?** The dictionary contains both raw keys. Class access returns the descriptor itself because our `__get__` implementations explicitly handle `instance is None`.

### Step 3: Read both names and inspect the call log

In [4]:
log01.clear()
print('locked =', h01.locked)
print('optional =', h01.optional)
print('events =', log01)

locked = validated-value
optional = raw-optional
events = ['data.get']


**What just happened?** The `locked` data descriptor runs and ignores the raw key; `optional` is shadowed by the dictionary, so its getter never runs.

### Step 4: Remove the shadow and read once more

In [5]:
del h01.__dict__['optional']
log01.clear()
print('optional without shadow =', h01.optional)
print('events =', log01)

optional without shadow = computed-value
events = ['nondata.get']


**What just happened?** Removing a shadowing key re-exposes the non-data descriptor. You did not remove or change the class descriptor itself.

### Complete solution

Build a diagnostic function that reads a name and returns both the visible result and the dispatch events.

In [6]:
def observed_read01(obj, name):
    log01.clear()
    value = getattr(obj, name)
    return value, tuple(log01)

h01.__dict__['optional'] = 'raw-again'
print(observed_read01(h01, 'locked'))
print(observed_read01(h01, 'optional'))

('validated-value', ('data.get',))
('raw-again', ())


**Why this solution works.** A read is explained by the **class definition plus instance state**, not by the dictionary alone. Instrumentation makes the difference observable.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [7]:
assert observed_read01(h01, 'locked') == ('validated-value', ('data.get',))
assert observed_read01(h01, 'optional') == ('raw-again', ())
del h01.__dict__['optional']
assert observed_read01(h01, 'optional') == ('computed-value', ('nondata.get',))
assert h01.__dict__['locked'] == 'raw-locked'
print('Problem 01: all checks passed')

Problem 01: all checks passed


**Test interpretation.** Notice that the raw data key is still present: the descriptor did not erase it; it merely took precedence over it.

### Takeaway and engineering note

Read precedence is an attribute-resolution policy, not an assertion that `__dict__` and descriptor storage cannot coexist. Direct writes to `__dict__` remain possible and can bypass validation.

---
# Problem 02 — The setter-only descriptor trap

**The challenge.** Implement a descriptor with `__set__` but **no `__get__`**. Determine what reading it returns before and after an instance key exists.

**What you will prove.** The precise boundary of data-descriptor precedence: having a setter alone does not manufacture a getter.

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Define a setter without defining a getter

In [8]:
class SetOnly02:
    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError('quantity must be an int')
        instance.__dict__['quantity'] = value

class Cart02:
    quantity = SetOnly02()

cart02 = Cart02()
raw02 = vars(Cart02)['quantity']
print('class has setter:', hasattr(type(raw02), '__set__'))
print('class has getter:', hasattr(type(raw02), '__get__'))

class has setter: True
class has getter: False


**What just happened?** `quantity` participates in managed **writes**. But it does not implement a callable getter, so we should not predict that `cart02.quantity` invokes one.

### Step 2: Read before the instance has any matching key

In [9]:
print('before write, read is descriptor:', cart02.quantity is raw02)
print('instance dictionary:', vars(cart02))

before write, read is descriptor: True
instance dictionary: {}


**What just happened?** When no instance entry exists, the descriptor object itself is the remaining class attribute returned by ordinary instance lookup.

### Step 3: Write normally, then inspect what a read finds

In [10]:
cart02.quantity = 7
print('after write:', cart02.quantity)
print('stored:', vars(cart02))

after write: 7
stored: {'quantity': 7}


**What just happened?** The setter wrote a key named `quantity`; because there is no `__get__` to invoke, the ordinary lookup now returns that dictionary entry.

### Step 4: Check an invalid assignment and a direct bypass

In [11]:
with expect_raises(TypeError):
    cart02.quantity = True  # bool is an int subclass; exact type is intentional here.
cart02.__dict__['quantity'] = 'bypassed'
print('bypassed read:', cart02.quantity)

bypassed read: bypassed


**What just happened?** The setter validates normal assignments but cannot intercept a direct dictionary mutation. The decision to reject `bool` is part of our *application contract*, not a rule of descriptor lookup.

### Complete solution

If you need managed reads as well as writes, implement `__get__` explicitly. Here a missing value raises a useful exception.

In [12]:
class CompleteQuantity02:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(f'{self.name} has not been initialized') from None

    def __set__(self, instance, value):
        if type(value) is not int or value < 0:
            raise ValueError('quantity must be a nonnegative integer')
        instance.__dict__[self.name] = value

class Inventory02:
    quantity = CompleteQuantity02()

inventory02 = Inventory02()
with expect_raises(AttributeError):
    _ = inventory02.quantity
inventory02.quantity = 8
print('managed read:', inventory02.quantity)

managed read: 8


**Why this solution works.** Now `__get__` is available and wins over a same-name instance key. Its body deliberately uses `instance.__dict__`, not `getattr(instance, self.name)`, to avoid recursive lookup.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [13]:
assert cart02.quantity == 'bypassed'  # The setter-only object has no getter.
assert inventory02.quantity == 8
inventory02.__dict__['quantity'] = 99
assert inventory02.quantity == 99  # The getter reads the same-name storage.
with expect_raises(ValueError):
    inventory02.quantity = -1
assert inventory02.quantity == 99
print('Problem 02: all checks passed')

Problem 02: all checks passed


**Test interpretation.** A complete descriptor can deliberately read a same-name instance key without being shadowed by that key. A setter-only descriptor has different read behavior.

### Takeaway and engineering note

Do not reduce the lookup rule to “anything with `__set__` always calls `__get__`.” A read can call `__get__` only if the found descriptor actually implements it.

---
# Problem 03 — A getter raises AttributeError — where does the error go?

**The challenge.** Show why accidental `AttributeError` inside a descriptor getter can trigger `__getattr__`, and choose an appropriate exception strategy for real failures.

**What you will prove.** The difference between absence, a broken getter, and fallback attributes (advanced extension).

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Create an intentionally broken getter

In [14]:
trace03 = []

class Fragile03:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        trace03.append('descriptor.get')
        raise AttributeError('internal backing field was unexpectedly missing')

class Service03:
    connection = Fragile03()

    def __getattr__(self, name):
        trace03.append(('fallback', name))
        if name == 'connection':
            return 'synthetic default'
        raise AttributeError(name)

**What just happened?** `__getattr__` is a fallback for attribute lookup failures. The failure may originate **inside** a getter, not just from an absent class name.

### Step 2: Perform an ordinary dot lookup

In [15]:
service03 = Service03()
print('value:', service03.connection)
print('trace:', trace03)

value: synthetic default
trace: ['descriptor.get', ('fallback', 'connection')]


**What just happened?** The getter raises `AttributeError`; ordinary lookup then uses `__getattr__`, which turns the failure into a synthetic value. This may accidentally hide a programming bug.

### Step 3: Bypass the normal fallback for diagnosis

In [16]:
trace03.clear()
with expect_raises(AttributeError):
    object.__getattribute__(service03, 'connection')
print('direct trace:', trace03)

direct trace: ['descriptor.get']


**What just happened?** Directly calling `object.__getattribute__` does not perform the outer dot-lookup fallback. It lets us see the original descriptor exception.

### Complete solution

Separate “value is intentionally unavailable” from “the getter failed unexpectedly.” In the latter case, use a non-`AttributeError` exception.

In [17]:
class SafeConnection03:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        raise RuntimeError('connection store unavailable')

class StrictService03:
    connection = SafeConnection03()

    def __getattr__(self, name):
        return f'fallback-for-{name}'

strict03 = StrictService03()
print('ordinary missing attribute:', strict03.unknown)
with expect_raises(RuntimeError):
    _ = strict03.connection

ordinary missing attribute: fallback-for-unknown


**Why this solution works.** `RuntimeError` signals a broken operation instead of a nonexistent attribute. If missing data truly means the attribute is unavailable, raising `AttributeError` may be intentional, but the caller must account for `__getattr__`.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [18]:
trace03.clear()
assert service03.connection == 'synthetic default'
assert trace03 == ['descriptor.get', ('fallback', 'connection')]
with expect_raises(RuntimeError):
    getattr(strict03, 'connection', 'default')
assert getattr(strict03, 'missing', 'default') == 'fallback-for-missing'
print('Problem 03: all checks passed')

Problem 03: all checks passed


**Test interpretation.** `getattr(obj, name, default)` does not suppress arbitrary exceptions; an actual `RuntimeError` stays visible.

### Takeaway and engineering note

A descriptor getter’s exception type is part of the attribute API. Avoid unintentionally converting internal errors into `AttributeError`, particularly when the owner defines `__getattr__`.

---
# Problem 04 — Why object.__setattr__ is not a validation escape hatch

**The challenge.** Test three mutation paths for an auditing descriptor: dot assignment, `object.__setattr__`, and a raw dictionary write.

**What you will prove.** How descriptor dispatch works for ordinary assignment versus intentional low-level bypasses.

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Write an auditing setter that stores per-instance values

In [19]:
audit04 = []

class AuditedScore04:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name, 0)

    def __set__(self, instance, value):
        if type(value) is not int or not 0 <= value <= 100:
            raise ValueError('score must be an integer in [0, 100]')
        old = instance.__dict__.get(self.name, 0)
        instance.__dict__[self.name] = value
        audit04.append((id(instance), old, value))

class Exam04:
    score = AuditedScore04()

**What just happened?** The descriptor writes only after validation, and keeps a log so we can detect which mutation routes actually execute `__set__`.

### Step 2: Use two seemingly different APIs for assignment

In [20]:
exam04 = Exam04()
exam04.score = 70
object.__setattr__(exam04, 'score', 75)
print('score:', exam04.score)
print('log transitions:', [(old, new) for _, old, new in audit04])

score: 75
log transitions: [(0, 70), (70, 75)]


**What just happened?** `object.__setattr__` is the *default attribute assignment machinery*: it still recognizes the data descriptor. It is not equivalent to directly editing `__dict__`.

### Step 3: Demonstrate the actual low-level bypass

In [21]:
exam04.__dict__['score'] = -500
print('read after bypass:', exam04.score)
print('log count:', len(audit04))

read after bypass: -500
log count: 2


**What just happened?** Our getter reads the raw storage, so bypassing the setter can insert invalid state. No audit entry was recorded. “Data descriptor wins reads” does not mean “instance dictionaries are tamper-proof.”

### Complete solution

Provide an explicit, validated restore method for external input, rather than populating `__dict__` from untrusted data.

In [22]:
class RestorableExam04:
    score = AuditedScore04()

    def __init__(self, initial):
        self.restore({'score': initial})

    def restore(self, payload):
        if set(payload) != {'score'}:
            raise ValueError('payload must contain exactly score')
        self.score = payload['score']

restored04 = RestorableExam04(80)
restored04.restore({'score': 91})
print('restored score:', restored04.score)

restored score: 91


**Why this solution works.** The restore method validates the payload shape and uses ordinary descriptor assignment, ensuring the value validation and audit run for both construction and restoration.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [23]:
assert [(old, new) for _, old, new in audit04[:2]] == [(0, 70), (70, 75)]
assert len(audit04) == 4  # two mutations above + initial construction + restore
assert exam04.score == -500
with expect_raises(ValueError):
    restored04.restore({'score': 101})
assert restored04.score == 91
with expect_raises(ValueError):
    restored04.restore({'score': 10, 'admin': True})
assert restored04.score == 91
print('Problem 04: all checks passed')

Problem 04: all checks passed


**Test interpretation.** Both assignment APIs route through `__set__`; direct dictionary changes do not. Validation failure leaves the previously valid value intact.

### Takeaway and engineering note

Use descriptors as an API-level validation boundary, not as a security barrier against code that can mutate an instance dictionary. Prefer explicit constructors and deserializers over `obj.__dict__.update(untrusted_payload)`.

---
# Problem 05 — Make “missing” distinct from None

**The challenge.** Design a managed setting for which `None` is a legitimate value, an unset value raises `AttributeError`, and deletion restores the unset state.

**What you will prove.** Correct sentinel usage, delete dispatch, class-level introspection, and testable state transitions.

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Observe the ambiguity in a tempting implementation

In [24]:
class Ambiguous05:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get('option', None)

class Demo05:
    option = Ambiguous05()

demo05 = Demo05()
print('missing:', demo05.option)
demo05.__dict__['option'] = None
print('explicitly set to None:', demo05.option)

missing: None
explicitly set to None: None


**What just happened?** The two states look identical. If the contract needs to distinguish them, `.get(..., None)` throws away information.

### Step 2: Introduce an identity-based sentinel

In [25]:
MISSING05 = object()

class OptionalValue05:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        value = instance.__dict__.get(self.name, MISSING05)
        if value is MISSING05:
            raise AttributeError(f'{self.name} is not initialized')
        return value

**What just happened?** The sentinel is compared with `is`, not `==`. A legitimate stored `None` remains distinguishable from an absent key.

### Step 3: Complete the mutation protocol

In [26]:
def optional_set05(self, instance, value):
    if value is not None and not isinstance(value, str):
        raise TypeError('expected string or None')
    instance.__dict__[self.name] = value

def optional_delete05(self, instance):
    try:
        del instance.__dict__[self.name]
    except KeyError:
        raise AttributeError(f'{self.name} is not initialized') from None

OptionalValue05.__set__ = optional_set05
OptionalValue05.__delete__ = optional_delete05

class Settings05:
    note = OptionalValue05()

settings05 = Settings05()
settings05.note = None
print('stored None:', settings05.note)

stored None: None


**What just happened?** Adding methods to the **descriptor class before creating `Settings05`** makes it a full data descriptor. In production, define the methods directly in the class; the temporary assignments here expose the transition step by step.

### Complete solution

Here is the complete, self-contained version to reuse in an application.

In [27]:
class NullableField05:
    _MISSING = object()

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        value = instance.__dict__.get(self.name, self._MISSING)
        if value is self._MISSING:
            raise AttributeError(f'{self.name} not set')
        return value

    def __set__(self, instance, value):
        if value is not None and not isinstance(value, str):
            raise TypeError(f'{self.name} must be str or None')
        instance.__dict__[self.name] = value

    def __delete__(self, instance):
        try:
            del instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(f'{self.name} not set') from None

class SettingsFinal05:
    note = NullableField05()

item05 = SettingsFinal05()
with expect_raises(AttributeError):
    _ = item05.note
item05.note = None
print('explicit None preserved:', item05.note)

explicit None preserved: None


**Why this solution works.** The read path explicitly distinguishes missing from `None`; assignment validates before modifying storage; deletion removes storage or signals absent state.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [28]:
assert item05.note is None
item05.note = 'ready'
assert item05.note == 'ready'
with expect_raises(TypeError):
    item05.note = 12
assert item05.note == 'ready'
del item05.note
with expect_raises(AttributeError):
    _ = item05.note
with expect_raises(AttributeError):
    del item05.note
assert SettingsFinal05.note is vars(SettingsFinal05)['note']
print('Problem 05: all checks passed')

Problem 05: all checks passed


**Test interpretation.** The invalid assignment does not corrupt the previous value, and class-level access returns the descriptor for introspection.

### Takeaway and engineering note

A clear missing-value contract is preferable to returning `None` for both “unset” and “explicitly set to None.” If callers can legitimately store your sentinel object, choose an unforgeable private sentinel strategy or encode presence separately.

---
# Problem 06 — One descriptor object, two attribute names: the aliasing bug

**The challenge.** Reuse a single descriptor instance for two fields and investigate why `__set_name__` can make both fields use one storage key. Repair the design without relying on accidental initialization order.

**What you will prove.** Descriptor objects are class-level shared objects; name metadata stored on the descriptor is shared, too (advanced extension).

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Write a descriptor that remembers its assigned name

In [29]:
class NamedField06:
    def __set_name__(self, owner, name):
        print('set_name:', owner.__name__, name)
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

shared06 = NamedField06()

**What just happened?** `__set_name__` stores one `name` property on **one** shared descriptor object. What happens if class creation assigns it to two different names?

### Step 2: Bind the same object twice

In [30]:
class BrokenRecord06:
    left = shared06
    right = shared06

broken06 = BrokenRecord06()
print('same descriptor object:', BrokenRecord06.left is BrokenRecord06.right)
print('recorded name:', shared06.name)

set_name: BrokenRecord06 left
set_name: BrokenRecord06 right
same descriptor object: True
recorded name: right


**What just happened?** Class creation calls `__set_name__` for each binding. The later call overwrites the descriptor object’s single `name` field. Both attributes now use the last recorded name.

### Step 3: Show the collision rather than merely describing it

In [31]:
broken06.left = 'L'
broken06.right = 'R'
print('stored:', vars(broken06))
print('two reads:', broken06.left, broken06.right)

stored: {'right': 'R'}
two reads: R R


**What just happened?** Both assignments target the same storage key. Even though the class has two attribute names, there is only one effective value.

### Step 4: Give each field its own descriptor instance

In [32]:
class GoodRecord06:
    left = NamedField06()
    right = NamedField06()

good06 = GoodRecord06()
good06.left = 'L'
good06.right = 'R'
print(vars(good06))
print(good06.left, good06.right)

set_name: GoodRecord06 left
set_name: GoodRecord06 right
{'left': 'L', 'right': 'R'}
L R


**What just happened?** Independent descriptor instances have independent `name` metadata. This is the simple, idiomatic solution when each binding represents a distinct field.

### Complete solution

For a library that might be misused, reject reuse explicitly during `__set_name__`. Depending on the interpreter and class-creation path, the original `TypeError` may be raised directly or wrapped; the example accepts either behavior.

In [33]:
class SingleBindingField06:
    def __set_name__(self, owner, name):
        if hasattr(self, '_bound_to'):
            raise TypeError(f'already bound to {self._bound_to}')
        self._bound_to = (owner.__name__, name)
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

single06 = SingleBindingField06()
try:
    class RejectedRecord06:
        first = single06
        second = single06
except (RuntimeError, TypeError) as exc:
    root_error = exc.__cause__ or exc
    print('class definition rejected:', type(root_error).__name__)

class definition rejected: TypeError


**Why this solution works.** The guard fails early instead of silently aliasing the state. The binding hook reports the original `TypeError`; the wrapper exception may vary by runtime. The demonstration handles both paths without assuming an identical traceback everywhere.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [34]:
assert vars(broken06) == {'right': 'R'}
assert (broken06.left, broken06.right) == ('R', 'R')
assert vars(good06) == {'left': 'L', 'right': 'R'}
assert GoodRecord06.left is not GoodRecord06.right
assert single06._bound_to == ('RejectedRecord06', 'first')
print('Problem 06: all checks passed')

Problem 06: all checks passed


**Test interpretation.** The broken case is intentional and documented; the corrected case stores two independent values. A defensive descriptor can make accidental reuse an immediate configuration error.

### Takeaway and engineering note

`__set_name__` is called for each binding during creation, not “once per unique descriptor object.” A field descriptor that saves one name should normally be instantiated separately for every field.

---
# Problem 07 — Inheritance does not call __set_name__ a second time

**The challenge.** Observe `__set_name__` on a base class, then inherit its descriptor through two subclasses without redefining it. Distinguish its original binding owner from the owner passed to later reads.

**What you will prove.** The lifecycle of descriptors across inheritance (advanced extension).

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Use a descriptor that records creation-time and access-time data

In [35]:
name_events07 = []
read_events07 = []

class OwnerProbe07:
    def __set_name__(self, owner, name):
        name_events07.append((owner.__name__, name))
        self.field_name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        read_events07.append((type(instance).__name__, owner.__name__))
        return instance.__dict__[self.field_name]

    def __set__(self, instance, value):
        instance.__dict__[self.field_name] = value

class Base07:
    token = OwnerProbe07()

print('creation events:', name_events07)

creation events: [('Base07', 'token')]


**What just happened?** The binding hook occurs when `Base07` is created. Inheritance itself does not create a new field object or trigger a new binding event.

### Step 2: Create two subclasses and check the creation log again

In [36]:
class Child07(Base07):
    pass

class Grandchild07(Child07):
    pass

print('creation events after inheritance:', name_events07)
print('one shared descriptor:', Base07.token is Child07.token is Grandchild07.token)

creation events after inheritance: [('Base07', 'token')]
one shared descriptor: True


**What just happened?** The original class descriptor is inherited. All three class lookups return the same descriptor because our class-access branch returns `self`.

### Step 3: Inspect the owner argument during instance reads

In [37]:
child07 = Child07()
grand07 = Grandchild07()
child07.token = 'child'
grand07.token = 'grand'
print(child07.token, grand07.token)
print('read events:', read_events07)

child grand
read events: [('Child07', 'Child07'), ('Grandchild07', 'Grandchild07')]


**What just happened?** For normal inherited instance lookup, the `owner` argument is the accessing instance’s type (the subclass), while the saved name remains the name assigned during the original class creation.

### Complete solution

Design a descriptor whose storage key is fixed at binding time but whose presentation depends on the dynamic `owner` argument.

In [38]:
class TaggedValue07:
    def __set_name__(self, owner, name):
        self.name = name
        self.defined_on = owner

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return (owner.__name__, instance.__dict__[self.name])

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

class Vehicle07:
    category = TaggedValue07()

class Bicycle07(Vehicle07):
    pass

bike07 = Bicycle07()
bike07.category = 'urban'
print('descriptor created on:', Bicycle07.category.defined_on.__name__)
print('instance read:', bike07.category)

descriptor created on: Vehicle07
instance read: ('Bicycle07', 'urban')


**Why this solution works.** The permanent binding metadata and the read-time owner serve different purposes; they should not be conflated or updated on every subclass access.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [39]:
assert name_events07 == [('Base07', 'token')]
assert (child07.token, grand07.token) == ('child', 'grand')
assert read_events07 == [('Child07', 'Child07'), ('Grandchild07', 'Grandchild07'),
                         ('Child07', 'Child07'), ('Grandchild07', 'Grandchild07')]
assert Vehicle07.category.defined_on is Vehicle07
assert bike07.category == ('Bicycle07', 'urban')
print('Problem 07: all checks passed')

Problem 07: all checks passed


**Test interpretation.** The read event log contains two events from the earlier `print` and two more from the assertions; inheritance itself added none to the name-binding log.

### Takeaway and engineering note

If a descriptor needs subclass-specific configuration, compute it using the `owner` passed to `__get__` or use an explicit registration scheme. Do not expect inherited descriptors to receive another `__set_name__` call.

---
# Problem 08 — A diamond MRO selects the descriptor BEFORE applying precedence

**The challenge.** Build two diamond-shaped inheritance orders whose branches define the same name differently. Confirm that an inherited data descriptor does not automatically win against a nearer class attribute in the MRO.

**What you will prove.** Two-stage lookup: first select the class attribute by MRO, then apply data/non-data and instance precedence.

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Define two competing branches

In [40]:
log08 = []

class Root08:
    pass

class ValidatingBranch08(Root08):
    class Checked:
        def __get__(self, instance, owner):
            if instance is None:
                return self
            log08.append('checked.get')
            return 'from descriptor'

        def __set__(self, instance, value):
            log08.append(('checked.set', value))
            instance.__dict__['field'] = value

    field = Checked()

class PlainBranch08(Root08):
    field = 'plain branch'

**What just happened?** `ValidatingBranch08.field` is a data descriptor; `PlainBranch08.field` is a string. They compete for the **same name** along separate branches.

### Step 2: Change only the order of bases

In [41]:
class CheckedFirst08(ValidatingBranch08, PlainBranch08):
    pass

class PlainFirst08(PlainBranch08, ValidatingBranch08):
    pass

print('checked first:', [cls.__name__ for cls in CheckedFirst08.__mro__])
print('plain first:', [cls.__name__ for cls in PlainFirst08.__mro__])

checked first: ['CheckedFirst08', 'ValidatingBranch08', 'PlainBranch08', 'Root08', 'object']
plain first: ['PlainFirst08', 'PlainBranch08', 'ValidatingBranch08', 'Root08', 'object']


**What just happened?** Both MROs are valid, but the first matching class `field` is different. Python does not search every ancestor and then globally select “the strongest descriptor.”

### Step 3: Insert matching instance keys in both objects

In [42]:
a08 = CheckedFirst08()
b08 = PlainFirst08()
a08.__dict__['field'] = 'instance A'
b08.__dict__['field'] = 'instance B'
print('checked-first read:', a08.field)
print('plain-first read:', b08.field)
print('trace:', log08)

checked-first read: from descriptor
plain-first read: instance B
trace: ['checked.get']


**What just happened?** The first MRO sees a data descriptor, which beats its instance key. The second MRO sees a plain class string, so its instance key wins.

### Step 4: Remove the instance key to expose the chosen class value

In [43]:
del b08.__dict__['field']
print('plain-first without instance shadow:', b08.field)
print('raw winner:', vars(PlainBranch08)['field'])

plain-first without instance shadow: plain branch
raw winner: plain branch


**What just happened?** The plain string is returned. The data descriptor in the other branch is not consulted because the MRO already found a nearer `field`.

### Complete solution

Build a small inspector for *which class defines the first matching name*; then reason about its descriptor category separately.

In [44]:
def first_defining_class08(cls, name):
    for candidate in cls.__mro__:
        if name in vars(candidate):
            return candidate, vars(candidate)[name]
    raise AttributeError(name)

for cls in (CheckedFirst08, PlainFirst08):
    defining_class, raw_attribute = first_defining_class08(cls, 'field')
    print(cls.__name__, '->', defining_class.__name__,
          '(data descriptor:', hasattr(type(raw_attribute), '__set__'), ')')

CheckedFirst08 -> ValidatingBranch08 (data descriptor: True )
PlainFirst08 -> PlainBranch08 (data descriptor: False )


**Why this solution works.** The helper deliberately reads class dictionaries rather than invoking descriptors, making the first stage of lookup explicit and side-effect-free.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [45]:
assert first_defining_class08(CheckedFirst08, 'field')[0] is ValidatingBranch08
assert first_defining_class08(PlainFirst08, 'field')[0] is PlainBranch08
assert a08.field == 'from descriptor'
assert b08.field == 'plain branch'
a08.field = 42
assert a08.__dict__['field'] == 42
assert a08.field == 'from descriptor'
print('Problem 08: all checks passed')

Problem 08: all checks passed


**Test interpretation.** The two-stage rule explains both the read and the write. The descriptor’s setter operates only in the hierarchy where it is the selected class attribute.

### Takeaway and engineering note

Always resolve the name in the MRO **first**. Data-descriptor precedence is applied to that selected object; it does not let a more distant ancestor overrule an earlier MRO definition.

---
# Problem 09 — A slot is itself a data descriptor

**The challenge.** Investigate why a subclass can have a dictionary key with the same name as an inherited slot without that key overriding the initialized slot.

**What you will prove.** How automatic slot member descriptors fit the same lookup mechanism (advanced extension).

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Let Python create a slot member descriptor

In [46]:
class SlottedBase09:
    __slots__ = ('code',)

    def __init__(self, code):
        self.code = code

slot_descriptor09 = vars(SlottedBase09)['code']
print('raw slot type:', type(slot_descriptor09).__name__)
print('get hook?', hasattr(type(slot_descriptor09), '__get__'))
print('set hook?', hasattr(type(slot_descriptor09), '__set__'))

raw slot type: member_descriptor
get hook? True
set hook? True


**What just happened?** A slot declaration creates an object in the class namespace that supplies descriptor get/set operations. It is not an ordinary instance dictionary key.

### Step 2: Show that the base instance has no dictionary

In [47]:
base09 = SlottedBase09('BASE')
print('base value:', base09.code)
print('has instance dictionary:', hasattr(base09, '__dict__'))

base value: BASE
has instance dictionary: False


**What just happened?** This base class only has the slot it declared. The lesson’s same-name `__dict__` storage approach cannot be used directly with this object.

### Step 3: Create a subclass with its default instance dictionary

In [48]:
class FlexibleChild09(SlottedBase09):
    pass

child09 = FlexibleChild09('REAL')
child09.__dict__['code'] = 'FAKE'
print('child dictionary:', vars(child09))
print('ordinary child read:', child09.code)

child dictionary: {'code': 'FAKE'}
ordinary child read: REAL


**What just happened?** The subclass has a dictionary, but the inherited slot descriptor is still a data descriptor; its getter wins against the injected same-name dictionary entry.

### Step 4: Delete the slot value and inspect the untouched dictionary

In [49]:
del child09.code
print('dictionary key still exists:', child09.__dict__['code'])
with expect_raises(AttributeError):
    _ = child09.code

dictionary key still exists: FAKE


**What just happened?** The slot getter raises `AttributeError` for an unset slot. Python does not continue to the shadowed dictionary key after a selected data descriptor getter fails.

### Complete solution

Use direct access to the raw slot descriptor only when debugging or implementing low-level tooling.

In [50]:
raw_slot09 = vars(SlottedBase09)['code']
raw_slot09.__set__(child09, 'RESTORED')
print('slot via explicit getter:', raw_slot09.__get__(child09, type(child09)))
print('ordinary attribute:', child09.code)
print('still-shadowed dictionary:', vars(child09))

slot via explicit getter: RESTORED
ordinary attribute: RESTORED
still-shadowed dictionary: {'code': 'FAKE'}


**Why this solution works.** The descriptor stores and retrieves the actual slot. Directly invoking its `__set__` is a diagnostic illustration, not a suggested replacement for normal `child09.code = ...`.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [51]:
assert not hasattr(base09, '__dict__')
assert vars(child09) == {'code': 'FAKE'}
assert child09.code == 'RESTORED'
child09.code = 'NORMAL'
assert child09.code == 'NORMAL'
assert vars(child09)['code'] == 'FAKE'
print('Problem 09: all checks passed')

Problem 09: all checks passed


**Test interpretation.** Slot storage and dictionary storage can coexist under the same visible name in a subclass, but the data descriptor determines the result of ordinary attribute access.

### Takeaway and engineering note

The `__slots__` limitation is about *available storage*, not about descriptor precedence disappearing. Slot member descriptors are a concrete, built-in example of the very rule studied in the lesson.

---
# Problem 10 — Compare classmethod and staticmethod under instance shadowing

**The challenge.** Explore the descriptor mechanics behind `classmethod` and `staticmethod`, then demonstrate how an instance key can override a normal instance read without changing class-level behavior.

**What you will prove.** Non-data descriptors can return very different objects while sharing the same precedence category (advanced extension).

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Create static and class methods with observable return values

In [52]:
class Factory10:
    @staticmethod
    def normalize(text):
        return text.strip().lower()

    @classmethod
    def identify(cls):
        return cls.__name__

class SpecializedFactory10(Factory10):
    pass

factory10 = SpecializedFactory10()
print('static via class:', Factory10.normalize('  HI  '))
print('class via subclass:', SpecializedFactory10.identify())

static via class: hi
class via subclass: SpecializedFactory10


**What just happened?** `staticmethod` exposes an underlying callable without automatic binding. `classmethod` binds the accessing class, which is why the inherited method reports the subclass name.

### Step 2: Inspect the raw class dictionary without invoking the wrappers

In [53]:
raw_static10 = vars(Factory10)['normalize']
raw_class10 = vars(Factory10)['identify']
print('raw types:', type(raw_static10).__name__, type(raw_class10).__name__)
print('get hooks:', hasattr(type(raw_static10), '__get__'),
      hasattr(type(raw_class10), '__get__'))
print('set hooks:', hasattr(type(raw_static10), '__set__'),
      hasattr(type(raw_class10), '__set__'))

raw types: staticmethod classmethod
get hooks: True True
set hooks: False False


**What just happened?** The wrapper objects supply `__get__`, but no `__set__`: they behave as non-data descriptors for ordinary instance attribute lookup.

### Step 3: Shadow both names on one instance

In [54]:
factory10.__dict__['normalize'] = lambda s: f'override:{s}'
factory10.__dict__['identify'] = lambda: 'instance override'
print('instance normalize:', factory10.normalize('HI'))
print('instance identify:', factory10.identify())
print('class normalize:', SpecializedFactory10.normalize(' HI '))
print('class identify:', SpecializedFactory10.identify())

instance normalize: override:HI
instance identify: instance override
class normalize: hi
class identify: SpecializedFactory10


**What just happened?** The injected instance functions are returned directly. Class-level lookup still uses the original descriptor objects because the instance dictionary belongs to one instance only.

### Step 4: Delete just one shadow

In [55]:
del factory10.__dict__['identify']
print('identify after deletion:', factory10.identify())
print('normalize still overridden:', factory10.normalize('HI'))

identify after deletion: SpecializedFactory10
normalize still overridden: override:HI


**What just happened?** Each name is independently shadowable. Removing a dictionary key restores inherited non-data-descriptor behavior for that name.

### Complete solution

When diagnosing method binding, inspect the wrapper directly and invoke its descriptor protocol explicitly.

In [56]:
class_bound10 = raw_class10.__get__(factory10, SpecializedFactory10)
static_bound10 = raw_static10.__get__(factory10, SpecializedFactory10)
print('manual class binding:', class_bound10())
print('manual static binding:', static_bound10(' HI '))

manual class binding: SpecializedFactory10
manual static binding: hi


**Why this solution works.** An explicit descriptor call shows that class-bound and unbound behavior is provided by the wrappers themselves, separate from attribute shadowing.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [57]:
assert factory10.normalize('HI') == 'override:HI'
assert factory10.identify() == 'SpecializedFactory10'
assert SpecializedFactory10.normalize(' HI ') == 'hi'
assert SpecializedFactory10.identify() == 'SpecializedFactory10'
assert class_bound10() == 'SpecializedFactory10'
assert static_bound10(' HI ') == 'hi'
print('Problem 10: all checks passed')

Problem 10: all checks passed


**Test interpretation.** All four access routes have well-defined outcomes. We distinguish wrapper behavior from lookup precedence instead of labeling every callable a “normal method.”

### Takeaway and engineering note

`staticmethod`, `classmethod`, and ordinary functions all implement descriptor binding differently; yet each is ordinarily shadowable on an instance because it is not a data descriptor.

---
# Problem 11 — Why len(obj) can disagree with obj.__len__()

**The challenge.** Put a non-data descriptor under a special-method name, then shadow that name in an instance dictionary. Compare explicit attribute access with implicit `len()` dispatch.

**What you will prove.** Implicit special-method lookup generally bypasses the instance dictionary and uses type-level dispatch (advanced extension).

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Make a descriptor that creates a zero-argument length function

In [58]:
length_events11 = []

class DynamicLength11:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        length_events11.append('descriptor.get')
        return lambda: len(instance.items)

class Basket11:
    __len__ = DynamicLength11()

    def __init__(self, items):
        self.items = list(items)

basket11 = Basket11(['a', 'b', 'c'])
print('explicit:', basket11.__len__())
print('implicit:', len(basket11))

explicit: 3
implicit: 3


**What just happened?** The non-data descriptor works in both contexts when no shadow is present. Note that special-method dispatch is not required to perform ordinary instance lookup.

### Step 2: Inject a conflicting instance attribute

In [59]:
basket11.__dict__['__len__'] = lambda: 1000
length_events11.clear()
print('explicit call:', basket11.__len__())
print('implicit call:', len(basket11))
print('descriptor getter events:', length_events11)

explicit call: 1000
implicit call: 3
descriptor getter events: ['descriptor.get']


**What just happened?** The explicit `basket11.__len__` uses ordinary lookup and finds the instance lambda. The implicit `len(basket11)` uses the class-level special method, invoking the descriptor instead.

### Step 3: Inspect the original descriptor through the class namespace

In [60]:
raw_len11 = vars(Basket11)['__len__']
print('raw descriptor type:', type(raw_len11).__name__)
print('raw method result:', raw_len11.__get__(basket11, Basket11)())
print('instance dictionary still holds:', basket11.__dict__['__len__']())

raw descriptor type: DynamicLength11
raw method result: 3
instance dictionary still holds: 1000


**What just happened?** The descriptor has not been overwritten. The divergence is caused by two different lookup entry points, not by inconsistent descriptor state.

### Complete solution

Make a diagnostic function that displays both lookup channels and clearly labels them.

In [61]:
def explain_length11(obj):
    return {
        'explicit_method_result': obj.__len__(),
        'implicit_len_result': len(obj),
        'instance_shadow': '__len__' in vars(obj),
    }

print(explain_length11(basket11))

{'explicit_method_result': 1000, 'implicit_len_result': 3, 'instance_shadow': True}


**Why this solution works.** The diagnostic records the discrepancy without claiming that the normal instance attribute precedence rule governs all Python operations.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [62]:
assert explain_length11(basket11) == {
    'explicit_method_result': 1000,
    'implicit_len_result': 3,
    'instance_shadow': True,
}
del basket11.__dict__['__len__']
assert basket11.__len__() == len(basket11) == 3
print('Problem 11: all checks passed')

Problem 11: all checks passed


**Test interpretation.** Deleting the instance shadow makes the explicit call use the same class descriptor again. The implicit `len()` value never depended on the instance shadow.

### Takeaway and engineering note

Avoid generalizing the ordinary `obj.name` lookup algorithm to implicit operations such as `len(obj)`, `obj + other`, or iteration. Special methods are normally resolved on the type.

---
# Problem 12 — Study functools.cached_property as a production non-data descriptor

**The challenge.** Use the standard library’s `cached_property` to demonstrate first-read calculation, same-name dictionary shadowing, reassignment, deletion, and a `__slots__` limitation.

**What you will prove.** Why non-data descriptors intentionally allow cached values to live under their own public name (advanced extension).

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Define a costly computation and count calls

In [63]:
class Report12:
    def __init__(self, numbers):
        self.numbers = list(numbers)
        self.calls = 0

    @cached_property
    def total(self):
        self.calls += 1
        return sum(self.numbers)

report12 = Report12([4, 5, 6])
print('before:', vars(report12))
print('first read:', report12.total)
print('after:', vars(report12))

before: {'numbers': [4, 5, 6], 'calls': 0}
first read: 15
after: {'numbers': [4, 5, 6], 'calls': 1, 'total': 15}


**What just happened?** The first getter call computes the result and saves it under `total` in the instance dictionary. The dictionary key then shadows the original non-data descriptor.

### Step 2: Verify that subsequent reads are not recomputed

In [64]:
report12.numbers.append(10)
print('cached total:', report12.total)
print('calls:', report12.calls)
print('raw class descriptor:', type(vars(Report12)['total']).__name__)

cached total: 15
calls: 1
raw class descriptor: cached_property


**What just happened?** Changing an input does not automatically invalidate a cached property. A cached result is a snapshot until you explicitly delete or overwrite it.

### Step 3: Reassign the cached public name, then delete it

In [65]:
report12.total = -1
print('user override:', report12.total)
del report12.total
print('recomputed after delete:', report12.total)
print('calls after invalidation:', report12.calls)

user override: -1
recomputed after delete: 25
calls after invalidation: 2


**What just happened?** A non-data descriptor does not intercept assignment, so a caller may replace the cached result. Deletion removes the shadowing key; the next read invokes the descriptor again.

### Step 4: Demonstrate the instance-dictionary requirement

In [66]:
class SlotOnlyReport12:
    __slots__ = ('numbers',)

    def __init__(self, numbers):
        self.numbers = list(numbers)

    @cached_property
    def total(self):
        return sum(self.numbers)

slotted_report12 = SlotOnlyReport12([1, 2, 3])
with expect_raises(TypeError):
    _ = slotted_report12.total
print('slot-only cached_property rejects missing __dict__')

slot-only cached_property rejects missing __dict__


**What just happened?** The standard `cached_property` needs a writable instance `__dict__`. A slot-only object that provides none raises `TypeError` rather than silently using shared class-level cache state.

### Complete solution

Expose a deliberate invalidation method so callers need not manipulate raw dictionaries themselves.

In [67]:
class MutableReport12:
    def __init__(self, numbers):
        self.numbers = list(numbers)
        self.compute_count = 0

    @cached_property
    def total(self):
        self.compute_count += 1
        return sum(self.numbers)

    def add(self, number):
        self.numbers.append(number)
        self.__dict__.pop('total', None)

mutable12 = MutableReport12([2, 3])
print('initial:', mutable12.total)
mutable12.add(7)
print('after add:', mutable12.total)

initial: 5
after add: 12


**Why this solution works.** The method owns both mutation and invalidation, reducing the chance of callers observing a stale derived value. Direct dictionary manipulation is contained in a documented internal implementation.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [68]:
assert report12.total == 25
assert report12.calls == 2
assert mutable12.total == 12
assert mutable12.compute_count == 2
assert 'total' in vars(mutable12)
assert not hasattr(slotted_report12, '__dict__')
print('Problem 12: all checks passed')

Problem 12: all checks passed


**Test interpretation.** The cache computes twice: once before the update and once afterward. It does not recompute every time it is read.

### Takeaway and engineering note

`cached_property` is a real-world use of non-data-descriptor shadowing. When freshness matters, document invalidation rules or use a different design; descriptors cannot automatically detect every mutation of their dependencies.

---
# Problem 13 — Class assignment is NOT an instance descriptor assignment

**The challenge.** Show that `obj.field = ...` calls the descriptor setter, while `Owner.field = ...` replaces the descriptor in the class dictionary. Then preserve the descriptor by intercepting class-level changes with a metaclass.

**What you will prove.** Separate instance attribute assignment from mutations of the class namespace (advanced extension).

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Define a guarded field on a normal class

In [69]:
changes13 = []

class Guarded13:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get('level', 0)

    def __set__(self, instance, value):
        if not isinstance(value, int):
            raise TypeError('level must be integer')
        instance.__dict__['level'] = value
        changes13.append(value)

class OrdinaryOwner13:
    level = Guarded13()

owner13 = OrdinaryOwner13()
owner13.level = 3
print('instance value:', owner13.level, 'setter log:', changes13)

instance value: 3 setter log: [3]


**What just happened?** Normal instance assignment finds and calls the data descriptor. The original object is still stored in the class dictionary.

### Step 2: Assign on the class itself and look for the original object

In [70]:
original13 = vars(OrdinaryOwner13)['level']
OrdinaryOwner13.level = 'replacement class value'
print('class now holds:', vars(OrdinaryOwner13)['level'])
print('instance still holds:', vars(owner13))
print('ordinary read after replacement:', owner13.level)

class now holds: replacement class value
instance still holds: {'level': 3}
ordinary read after replacement: 3


**What just happened?** Class assignment changed the class dictionary: it did **not** invoke `Guarded13.__set__` on an instance. The old instance entry is now read directly because the class descriptor is gone.

### Step 3: Notice that the old object remains callable if saved elsewhere

In [71]:
print('saved descriptor read:', original13.__get__(owner13, OrdinaryOwner13))
print('same class slot?', vars(OrdinaryOwner13)['level'] is original13)

saved descriptor read: 3
same class slot? False


**What just happened?** Replacing the class binding removes descriptor dispatch from ordinary lookup, but it does not destroy other references to the original descriptor object.

### Complete solution

Use a metaclass that rejects replacement of an existing protected descriptor name. This is an API guard, not a security boundary against arbitrary low-level code.

In [72]:
class ProtectDescriptors13(type):
    def __setattr__(cls, name, value):
        previous = cls.__dict__.get(name)
        if previous is not None and getattr(type(previous), '__set__', None) is not None:
            raise AttributeError(f'cannot replace descriptor {name!r}')
        super().__setattr__(name, value)

class ProtectedOwner13(metaclass=ProtectDescriptors13):
    level = Guarded13()

protected13 = ProtectedOwner13()
protected13.level = 5
with expect_raises(AttributeError):
    ProtectedOwner13.level = 'accident'
print('protected instance value:', protected13.level)

protected instance value: 5


**Why this solution works.** The metaclass intercepts assignments made to the **class object**, whereas `Guarded13.__set__` intercepts assignments made to the **instance**. The two interception points are different.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [73]:
assert changes13 == [3, 5]
assert owner13.level == 3
assert OrdinaryOwner13.level == 'replacement class value'
assert ProtectedOwner13.level is vars(ProtectedOwner13)['level']
with expect_raises(TypeError):
    protected13.level = 'bad'
assert protected13.level == 5
print('Problem 13: all checks passed')

Problem 13: all checks passed


**Test interpretation.** After the normal class is modified, its instance reads use dictionary storage directly. The protected class retains the original validating descriptor.

### Takeaway and engineering note

The data-descriptor protocol targets **instance** mutation. Library authors who must control modifications to class definitions need separate class-level mechanisms, such as a metaclass or explicit configuration API.

---
# Problem 14 — Special descriptor methods are looked up on the descriptor TYPE

**The challenge.** Try adding `__set__` to one descriptor *instance*, then to its class. Explain why only the second operation changes the host object’s descriptor classification and assignment behavior.

**What you will prove.** Special-method lookup and the fact that descriptor classification uses type-level slots (advanced extension).

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Begin with a minimal non-data descriptor

In [74]:
events14 = []

class Evolving14:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        events14.append('getter')
        return 'from getter'

probe14 = Evolving14()

class Container14:
    value = probe14

container14 = Container14()
container14.__dict__['value'] = 'raw initial'
print('initial read:', container14.value)

initial read: raw initial


**What just happened?** Because `Evolving14` has only a getter, a raw instance key wins. We will test whether assigning a magic-looking method on the descriptor *object* changes that.

### Step 2: Attach a setter function to the descriptor instance itself

In [75]:
def installed_setter14(self, instance, value):
    events14.append(('setter', value))
    instance.__dict__['value'] = f'managed:{value}'

probe14.__set__ = installed_setter14
print('instance advertises __set__:', hasattr(probe14, '__set__'))
print('type advertises __set__:', hasattr(type(probe14), '__set__'))
container14.value = 'assigned before type patch'
print('stored:', vars(container14))

instance advertises __set__: True
type advertises __set__: False
stored: {'value': 'assigned before type patch'}


**What just happened?** Special descriptor hooks are not discovered by arbitrary instance-attribute lookup on the descriptor object. The host still treats this as a non-data descriptor, and assignment writes the host instance dictionary.

### Step 3: Attach the same method to the descriptor class

In [76]:
Evolving14.__set__ = installed_setter14
print('type advertises __set__:', hasattr(type(probe14), '__set__'))
print('read after type patch:', container14.value)
container14.value = 'assigned after type patch'
print('stored after patched setter:', vars(container14))
print('event tail:', events14[-2:])

type advertises __set__: True
read after type patch: from getter
stored after patched setter: {'value': 'managed:assigned after type patch'}
event tail: ['getter', ('setter', 'assigned after type patch')]


**What just happened?** Once the *type* has `__set__`, the same descriptor object becomes a data descriptor. Its getter wins on reads, and new assignments invoke the injected class-level setter.

### Step 4: Restore the type to its original definition

In [77]:
del Evolving14.__set__
print('back to non-data read:', container14.value)
print('setter event count:', sum(isinstance(x, tuple) for x in events14))

back to non-data read: managed:assigned after type patch
setter event count: 1


**What just happened?** Removing the class-level special method restores the original non-data behavior. The previously injected `probe14.__set__` attribute remains on the object but is irrelevant to dispatch.

### Complete solution

In production, declare the full descriptor protocol in the descriptor class definition. The dynamic modifications above are diagnostic experiments, not a recommended runtime pattern.

In [78]:
class StableManaged14:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get('value', '<missing>')

    def __set__(self, instance, value):
        instance.__dict__['value'] = f'managed:{value}'

class StableContainer14:
    value = StableManaged14()

stable14 = StableContainer14()
stable14.value = 'hello'
print('stable value:', stable14.value)

stable value: managed:hello


**Why this solution works.** Declaring methods on the type at definition time makes lookup semantics stable and reviewable, with no global behavior changes midway through application execution.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [79]:
assert container14.value == 'managed:assigned after type patch'
assert ('setter', 'assigned after type patch') in events14
assert not hasattr(Evolving14, '__set__')
assert hasattr(probe14, '__set__')  # Still not a descriptor protocol method.
assert stable14.value == 'managed:hello'
print('Problem 14: all checks passed')

Problem 14: all checks passed


**Test interpretation.** An instance attribute named `__set__` does not supply the descriptor protocol; a type-level method does. The cleaned-up final class has the intended behavior from the start.

### Takeaway and engineering note

When introspecting descriptor categories, examine methods on `type(descriptor)`, not `descriptor.__dict__`. Avoid monkey-patching descriptor classes in libraries: one change may affect every object of that descriptor type.

---
# Problem 15 — Same-name storage is a choice, not a requirement

**The challenge.** Build a data descriptor that stores values under a private backing key and compare it with the lesson’s same-name storage strategy. Attempt a same-name dictionary injection and predict the result.

**What you will prove.** Descriptor precedence can hide a dictionary key *even when the getter never uses that key*.

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Choose separate public and backing names

In [80]:
class BackedField15:
    def __set_name__(self, owner, name):
        self.public_name = name
        self.storage_name = f'_{name}_validated'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.storage_name]
        except KeyError:
            raise AttributeError(f'{self.public_name} is unset') from None

    def __set__(self, instance, value):
        if not isinstance(value, str) or not value:
            raise ValueError(f'{self.public_name} needs a nonempty string')
        instance.__dict__[self.storage_name] = value

class Customer15:
    username = BackedField15()

**What just happened?** Unlike the source example’s `instance.__dict__[self.prop_name]` approach, this design deliberately stores the validated value under a different key. Both techniques are possible for data descriptors.

### Step 2: Perform a normal assignment and inspect all dictionary keys

In [81]:
customer15 = Customer15()
customer15.username = 'ada'
print('read:', customer15.username)
print('dictionary:', vars(customer15))

read: ada
dictionary: {'_username_validated': 'ada'}


**What just happened?** The public `username` attribute resolves through the data descriptor, which reads `_username_validated`. No public-name entry is needed for this design.

### Step 3: Insert a fake value under the public name

In [82]:
customer15.__dict__['username'] = 'FAKE'
print('dictionary now:', vars(customer15))
print('ordinary read:', customer15.username)

dictionary now: {'_username_validated': 'ada', 'username': 'FAKE'}
ordinary read: ada


**What just happened?** The injected key does not win against a data descriptor. More importantly, the getter reads a **different** key, so the fake string has no effect on the returned value.

### Step 4: Contrast this with same-name storage in a second descriptor

In [83]:
class SameName15(BackedField15):
    def __set_name__(self, owner, name):
        self.public_name = name
        self.storage_name = name

class AlternativeCustomer15:
    username = SameName15()

alternative15 = AlternativeCustomer15()
alternative15.username = 'ada'
alternative15.__dict__['username'] = 'FAKE'
print('same-name result:', alternative15.username)

same-name result: FAKE


**What just happened?** Both are data descriptors. The source-inspired same-name getter correctly runs, but it reads a key that raw dictionary mutation can replace. Private backing avoids *that specific collision*, not all possible bypasses.

### Complete solution

Document storage choice as part of a descriptor’s contract, and use a stable `__set_name__`-derived backing key for each field.

In [84]:
class LabeledCustomer15:
    username = BackedField15()
    display_name = BackedField15()

labeled15 = LabeledCustomer15()
labeled15.username = 'ada'
labeled15.display_name = 'Ada Lovelace'
print('public fields:', labeled15.username, labeled15.display_name)
print('internal backing keys:', sorted(vars(labeled15)))

public fields: ada Ada Lovelace
internal backing keys: ['_display_name_validated', '_username_validated']


**Why this solution works.** The backing names are derived from unique public attribute names, so two descriptor instances operate on independent dictionary keys. The getter and setter agree on their storage convention.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [85]:
assert customer15.username == 'ada'
assert vars(customer15)['username'] == 'FAKE'
assert alternative15.username == 'FAKE'
assert vars(labeled15) == {
    '_username_validated': 'ada',
    '_display_name_validated': 'Ada Lovelace',
}
with expect_raises(ValueError):
    labeled15.username = ''
assert labeled15.username == 'ada'
print('Problem 15: all checks passed')

Problem 15: all checks passed


**Test interpretation.** Separate backing storage keeps raw public-name injection from changing the property’s returned value, while ordinary invalid assignments remain atomic.

### Takeaway and engineering note

The original lesson’s same-name storage is convenient precisely because a data descriptor cannot be shadowed. A distinct backing key is also valid; choose based on compatibility, introspection, and invariants, and do not mistake either for protection from arbitrary dictionary edits.

---
# Problem 16 — Build an atomic, descriptor-driven configuration importer

**The challenge.** Implement two validating data descriptors and a safe `apply(mapping)` operation that updates *both* fields only after all validation succeeds. Use no automatic third-party model framework.

**What you will prove.** Design a class that respects descriptor validation and an all-or-nothing state-update contract (advanced capstone).

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Start with a field that validates without writing

In [86]:
class PositiveInt16:
    def __set_name__(self, owner, name):
        self.name = name

    def validate(self, value):
        if type(value) is not int or value <= 0:
            raise ValueError(f'{self.name} must be a positive integer')
        return value

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        instance.__dict__[self.name] = self.validate(value)

**What just happened?** By extracting a pure `validate(value)` method, we can check a proposed multi-field update *before* changing any instance state. This avoids a partial update when the second field is invalid.

### Step 2: Define a normalized string field with the same interface

In [87]:
class NonBlank16:
    def __set_name__(self, owner, name):
        self.name = name

    def validate(self, value):
        if not isinstance(value, str):
            raise TypeError(f'{self.name} must be a string')
        normalized = value.strip()
        if not normalized:
            raise ValueError(f'{self.name} cannot be blank')
        return normalized

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        instance.__dict__[self.name] = self.validate(value)

**What just happened?** The two descriptors share a tiny validation interface but enforce different rules. Both use the lesson’s safe direct dictionary storage under the property name.

### Step 3: Demonstrate the dangerous sequential-update approach

In [88]:
class NaiveConfig16:
    workers = PositiveInt16()
    label = NonBlank16()

naive16 = NaiveConfig16()
naive16.workers = 2
naive16.label = 'old'
try:
    naive16.workers = 8
    naive16.label = '   '
except ValueError:
    pass
print('partially updated:', vars(naive16))

partially updated: {'workers': 8, 'label': 'old'}


**What just happened?** Each descriptor independently validates its own field. The first update succeeded before the second raised, so sequential assignments did not provide transaction semantics.

### Step 4: Validate all proposed values before committing

In [89]:
class Config16:
    workers = PositiveInt16()
    label = NonBlank16()

    _fields = ('workers', 'label')

    def __init__(self, workers, label):
        self.apply({'workers': workers, 'label': label})

    def apply(self, mapping):
        if set(mapping) != set(self._fields):
            raise ValueError('supply exactly workers and label')
        validated = {}
        for name in self._fields:
            descriptor = vars(type(self))[name]
            validated[name] = descriptor.validate(mapping[name])
        # Once ALL values have passed validation, commit them using setters.
        for name in self._fields:
            setattr(self, name, validated[name])

config16 = Config16(3, '  production  ')
print('initial:', vars(config16))

initial: {'workers': 3, 'label': 'production'}


**What just happened?** The complete proposed update is validated first. After that, both assignment operations see valid, already-normalized values. The design assumes validators are deterministic and setters have no additional failure-prone side effects.

### Complete solution

Turn invalid updates into explicit regression tests, and provide a method for constructing a validated object from a mapping.

In [90]:
@classmethod
def from_mapping16(cls, mapping):
    if set(mapping) != set(cls._fields):
        raise ValueError('unexpected configuration keys')
    return cls(mapping['workers'], mapping['label'])

Config16.from_mapping = from_mapping16
copy16 = Config16.from_mapping({'workers': 6, 'label': '  staging '})
print('imported:', vars(copy16))

before16 = vars(config16).copy()
with expect_raises(ValueError):
    config16.apply({'workers': 10, 'label': '  '})
print('after failed update:', vars(config16))

imported: {'workers': 6, 'label': 'staging'}
after failed update: {'workers': 3, 'label': 'production'}


**Why this solution works.** The `from_mapping` constructor delegates to the validated constructor, and `apply` does not write either value until all validations succeed. It avoids the unsafe `__dict__.update(mapping)` shortcut.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [91]:
assert vars(naive16) == {'workers': 8, 'label': 'old'}
assert vars(config16) == before16 == {'workers': 3, 'label': 'production'}
assert vars(copy16) == {'workers': 6, 'label': 'staging'}
with expect_raises(ValueError):
    config16.apply({'workers': -1, 'label': 'works'})
with expect_raises(TypeError):
    config16.apply({'workers': 9, 'label': None})
with expect_raises(ValueError):
    config16.apply({'workers': 5})
assert vars(config16) == before16
config16.apply({'workers': 4, 'label': '  blue  '})
assert vars(config16) == {'workers': 4, 'label': 'blue'}
print('Problem 16: all checks passed')

Problem 16: all checks passed


**Test interpretation.** Invalid type, invalid content, incomplete schema, and an attempted two-field update all leave the valid state unchanged. A valid input updates and normalizes both fields.

### Takeaway and engineering note

Descriptor validation governs one assignment at a time. For multi-field invariants and imports, perform a separate validation phase and define an explicit commit strategy; beware of custom setters with non-idempotent side effects or concurrent mutation.

---
# Problem 17 — Descriptors, shallow copies, and accidental mutable aliases

**The challenge.** Store list values through a managed data descriptor, copy the owning instance, and determine which state is independent and which state remains shared. Provide a controlled repair.

**What you will prove.** Descriptor state is per instance, but mutable values copied shallowly may still alias (advanced extension).

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Write a descriptor that validates an incoming list

In [92]:
import copy

class StringList17:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, values):
        if not isinstance(values, list) or not all(isinstance(x, str) for x in values):
            raise TypeError('expected a list of strings')
        instance.__dict__[self.name] = list(values)  # Copy the caller's list.

class Playlist17:
    songs = StringList17()

    def __init__(self, songs):
        self.songs = songs

input17 = ['A', 'B']
original17 = Playlist17(input17)
input17.append('C')
print('caller input:', input17)
print('stored:', original17.songs)

caller input: ['A', 'B', 'C']
stored: ['A', 'B']


**What just happened?** Copying on assignment prevents the caller from modifying the model through its *original input list*. It does not automatically prevent every other form of mutable aliasing.

### Step 2: Make a shallow copy of the owner

In [93]:
shallow17 = copy.copy(original17)
print('different objects:', shallow17 is not original17)
print('different dictionaries:', shallow17.__dict__ is not original17.__dict__)
print('same nested list:', shallow17.songs is original17.songs)

different objects: True
different dictionaries: True
same nested list: True


**What just happened?** The owners and their dictionaries are different, but both dictionaries point to the same nested list. Descriptors do not control how `copy.copy` duplicates arbitrary values.

### Step 3: Mutate the shared list through the copied object

In [94]:
shallow17.songs.append('D')
print('original after mutation:', original17.songs)
print('shallow after mutation:', shallow17.songs)

original after mutation: ['A', 'B', 'D']
shallow after mutation: ['A', 'B', 'D']


**What just happened?** The mutation happens *inside* a list, not through `Playlist17.songs.__set__`. Both objects observe it because their list references are shared.

### Step 4: Compare this with a deep copy

In [95]:
deep17 = copy.deepcopy(original17)
deep17.songs.append('E')
print('deep:', deep17.songs)
print('original:', original17.songs)
print('list identities differ:', deep17.songs is not original17.songs)

deep: ['A', 'B', 'D', 'E']
original: ['A', 'B', 'D']
list identities differ: True


**What just happened?** A deep copy duplicates the nested list in this simple example. The appropriate policy depends on the intended semantics of your own class and its contained objects.

### Complete solution

For a stricter API, return a read-only tuple and accept any finite iterable that has been materialized and validated. This eliminates list mutation through the descriptor read.

In [96]:
class ImmutableTags17:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, values):
        materialized = tuple(values)
        if not all(isinstance(x, str) for x in materialized):
            raise TypeError('all tags must be strings')
        instance.__dict__[self.name] = materialized

class ImmutablePlaylist17:
    tags = ImmutableTags17()

    def __init__(self, tags):
        self.tags = tags

safe17 = ImmutablePlaylist17(['jazz', 'live'])
safe_copy17 = copy.copy(safe17)
print('immutable tags:', safe17.tags)

immutable tags: ('jazz', 'live')


**Why this solution works.** Tuples cannot be mutated in place. The new design still permits whole-attribute reassignment, which runs validation, but avoids exposing an internal mutable list for uncontrolled mutation.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [97]:
assert original17.songs == ['A', 'B', 'D']
assert shallow17.songs is original17.songs
assert deep17.songs == ['A', 'B', 'D', 'E']
assert deep17.songs is not original17.songs
assert safe17.tags == ('jazz', 'live')
assert safe_copy17.tags == safe17.tags
with expect_raises(AttributeError):
    safe17.tags.append('other')
with expect_raises(TypeError):
    safe17.tags = ('valid', 123)
assert safe17.tags == ('jazz', 'live')
print('Problem 17: all checks passed')

Problem 17: all checks passed


**Test interpretation.** The failures and successes distinguish assignment validation, mutations of stored objects, and copy behavior. These are three separate responsibilities.

### Takeaway and engineering note

“Per-instance storage” only guarantees separate top-level storage locations. It does not ensure that mutable objects referenced by different instances are independent. Prefer immutable exposed values or define explicit copy policies for mutable domain models.

---
# Problem 18 — Final investigation: predict a complete dispatch transcript

**The challenge.** Build a small class containing a data descriptor, a non-data descriptor, and a normal class constant. Predict the exact values and descriptor-call log after a series of mutations, including a class-level replacement.

**What you will prove.** Synthesize MRO selection, data precedence, non-data shadowing, and class namespace mutation in an executable investigation.

In the next cells, make the prediction before running the code. The worked answer is developed incrementally; the final assertions are executable acceptance tests.

### Step 1: Define all three sources of attributes

In [98]:
transcript18 = []

class Data18:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        transcript18.append('data.get')
        return instance.__dict__['primary'] * 10

    def __set__(self, instance, value):
        transcript18.append(('data.set', value))
        instance.__dict__['primary'] = value

class NonData18:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        transcript18.append('nondata.get')
        return 'computed'

class Puzzle18:
    primary = Data18()
    secondary = NonData18()
    category = 'class default'

puzzle18 = Puzzle18()
puzzle18.primary = 4
transcript18.clear()

**What just happened?** The data descriptor multiplies its stored value on read. The non-data descriptor computes a constant, and `category` is a plain class value. The setup clears the setter event so the read transcript is easy to interpret.

### Step 2: Give every name a raw dictionary collision

In [99]:
puzzle18.__dict__.update(primary=5, secondary='raw secondary',
                         category='raw category')
print('raw dictionary:', vars(puzzle18))
print('read triple:', (puzzle18.primary, puzzle18.secondary, puzzle18.category))
print('trace:', transcript18)

raw dictionary: {'primary': 5, 'secondary': 'raw secondary', 'category': 'raw category'}
read triple: (50, 'raw secondary', 'raw category')
trace: ['data.get']


**What just happened?** Only the data descriptor getter runs: `primary` produces 50; the non-data descriptor and ordinary class value both lose to matching instance keys.

### Step 3: Remove the shadow on the non-data descriptor only

In [100]:
del puzzle18.__dict__['secondary']
transcript18.clear()
print('secondary read:', puzzle18.secondary)
print('trace:', transcript18)

secondary read: computed
trace: ['nondata.get']


**What just happened?** With its shadow removed, the non-data descriptor getter is visible again. The data descriptor has not been consulted in this step.

### Step 4: Replace the data descriptor on the CLASS

In [101]:
old_primary18 = vars(Puzzle18)['primary']
Puzzle18.primary = 'new class default'
transcript18.clear()
print('primary after class replacement:', puzzle18.primary)
print('trace after replacement:', transcript18)
print('saved old descriptor result:', old_primary18.__get__(puzzle18, Puzzle18))

primary after class replacement: 5
trace after replacement: []
saved old descriptor result: 50


**What just happened?** The `Puzzle18.primary` rebinding removes the descriptor from ordinary lookup, so the instance key `primary=5` now wins. Calling the saved descriptor manually still computes 50 and logs an event.

### Complete solution

Encode the final state as a reusable, explicit answer key: distinguish an ordinary class assignment from an instance-level access.

In [102]:
answer18 = {
    'instance_primary': puzzle18.primary,
    'instance_secondary': puzzle18.secondary,
    'instance_category': puzzle18.category,
    'class_primary': Puzzle18.primary,
    'raw_primary_key': vars(puzzle18)['primary'],
}
print('answer key:', answer18)
print('final transcript:', transcript18)

answer key: {'instance_primary': 5, 'instance_secondary': 'computed', 'instance_category': 'raw category', 'class_primary': 'new class default', 'raw_primary_key': 5}
final transcript: ['data.get', 'nondata.get']


**Why this solution works.** A correct explanation begins with *which class attribute currently exists*, checks whether a matching instance key is present, and only then decides whether `__get__` will execute.

### Verify the contract

These assertions test observable behavior, including the boundary cases that are easy to overlook.

In [103]:
assert answer18 == {
    'instance_primary': 5,
    'instance_secondary': 'computed',
    'instance_category': 'raw category',
    'class_primary': 'new class default',
    'raw_primary_key': 5,
}
assert transcript18 == ['data.get', 'nondata.get']
assert vars(Puzzle18)['primary'] is not old_primary18
assert old_primary18.__get__(puzzle18, Puzzle18) == 50
assert transcript18[-1] == 'data.get'
print('Problem 18: all checks passed')

Problem 18: all checks passed


**Test interpretation.** The transcript contains one direct old-descriptor call from Step 4 and one non-data getter from the answer-key construction; the final explicit getter call adds the last event. No invisible descriptor calls are assumed.

### Takeaway and engineering note

A reliable descriptor diagnosis has four questions: (1) what does the MRO currently select? (2) is that object a data descriptor with a getter? (3) does the instance contain the name? (4) will the selected descriptor or fallback execute? Re-evaluate from scratch after mutations to the class or instance.

---
# End-of-notebook review

You have worked through **18 new, tutorial-paced advanced problems**, each with a challenge, incremental experiments, explanatory cells, a worked solution, and executable checks.

The original supplied lesson emphasizes the *same-name storage* observation and the danger of recursive `setattr` inside a descriptor. This second collection preserves those foundations while exploring less obvious edges: setter-only objects, `AttributeError` fallbacks, instance versus class mutation, descriptor aliasing, inherited binding metadata, diamond MRO, slot member descriptors, class/static methods, implicit special-method lookup, standard caching, type-level protocol changes, transactional imports, and mutable aliasing under copying.

**Suggested next exercise:** without looking back, implement a descriptor storing a nonempty string, supporting deletion, working through two levels of inheritance, and emitting a small event log. Predict what will happen if you insert a same-name key into the dictionary and then replace the class binding. Check your prediction with assertions.

**Best-practice recap:** use `__set_name__` to record field names; return the descriptor on class access; use direct backing storage inside getters/setters rather than `getattr`/`setattr` on the same public field; validate before committing; test invalid values and missing values; and explicitly document what direct `__dict__` access, `__slots__`, class monkey-patching, and mutable nested state can do to your guarantees.